In [3]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')


df = pd.read_csv('/content/drive/MyDrive/rapido_project/data/captains_scored.csv')
print(df.shape)
print(df['risk_segment'].value_counts())

Mounted at /content/drive
(10000, 33)
risk_segment
Low Risk       3729
Medium Risk    3323
High Risk      2948
Name: count, dtype: int64


In [4]:
# ================================================
# RAPIDO CAPTAIN RETENTION ROI CALCULATOR
# ================================================


avg_rides_per_day = 10
avg_fare_per_ride = 90
platform_commission = 0.18
revenue_per_captain_per_day = avg_rides_per_day * avg_fare_per_ride * platform_commission

days_retained = 30
revenue_per_saved_captain = revenue_per_captain_per_day * days_retained

bonus_amount = 200
ops_cost_per_captain = 50
total_intervention_cost = bonus_amount + ops_cost_per_captain

high_risk_captains = len(df[df['risk_segment'] == 'High Risk'])
intervention_success_rate = 0.30
actual_churners_in_high_risk = int(high_risk_captains * 0.68)
captains_saved = int(actual_churners_in_high_risk * intervention_success_rate)

total_spend = high_risk_captains * total_intervention_cost
revenue_recovered = captains_saved * revenue_per_saved_captain
net_roi = revenue_recovered - total_spend
roi_pct = (net_roi / total_spend) * 100

print("=" * 55)
print("   RAPIDO CAPTAIN RETENTION — ROI CALCULATOR")
print("=" * 55)
print(f"\nHigh Risk Captains identified     : {high_risk_captains:,}")
print(f"Estimated actual churners         : {actual_churners_in_high_risk:,}")
print(f"Captains saved (30% success rate) : {captains_saved:,}")
print(f"\nRevenue per saved captain (30d)   : Rs.{revenue_per_saved_captain:,.0f}")
print(f"Total revenue recovered           : Rs.{revenue_recovered:,.0f}")
print(f"\nTotal intervention spend          : Rs.{total_spend:,.0f}")
print(f"Net ROI                           : Rs.{net_roi:,.0f}")
print(f"ROI percentage                    : {roi_pct:.0f}%")
print("=" * 55)

   RAPIDO CAPTAIN RETENTION — ROI CALCULATOR

High Risk Captains identified     : 2,948
Estimated actual churners         : 2,004
Captains saved (30% success rate) : 601

Revenue per saved captain (30d)   : Rs.4,860
Total revenue recovered           : Rs.2,920,860

Total intervention spend          : Rs.737,000
Net ROI                           : Rs.2,183,860
ROI percentage                    : 296%


In [5]:
print("=" * 55)
print("   COST OF NOT INTERVENING")
print("=" * 55)

# What Rapido loses if they ignore high risk captains
churners_lost = actual_churners_in_high_risk
revenue_lost = churners_lost * revenue_per_saved_captain
acquisition_cost_per_captain = 500   # Rs. to onboard a new captain

replacement_cost = churners_lost * acquisition_cost_per_captain
total_loss = revenue_lost + replacement_cost

print(f"\nCaptains lost without intervention : {churners_lost:,}")
print(f"Revenue lost (30 days)             : Rs.{revenue_lost:,.0f}")
print(f"Replacement/acquisition cost       : Rs.{replacement_cost:,.0f}")
print(f"Total business loss                : Rs.{total_loss:,.0f}")
print(f"\nIntervention spend was only        : Rs.{total_spend:,.0f}")
print(f"You save Rs.{total_loss - total_spend:,.0f} by acting early")
print("=" * 55)

   COST OF NOT INTERVENING

Captains lost without intervention : 2,004
Revenue lost (30 days)             : Rs.9,739,440
Replacement/acquisition cost       : Rs.1,002,000
Total business loss                : Rs.10,741,440

Intervention spend was only        : Rs.737,000
You save Rs.10,004,440 by acting early


In [6]:
print("=" * 60)
print("   RECOMMENDED INTERVENTION STRATEGY")
print("=" * 60)

strategy = {
    'High Risk (68% churn)': {
        'count': len(df[df['risk_segment'] == 'High Risk']),
        'action': 'Immediate Rs.200 bonus + personal outreach call',
        'trigger': 'Week 4 rides < 8 AND parttime AND bike',
        'urgency': 'Within 48 hours'
    },
    'Medium Risk (43% churn)': {
        'count': len(df[df['risk_segment'] == 'Medium Risk']),
        'action': 'Push notification with streak bonus offer',
        'trigger': 'Ride decay > 40% week over week',
        'urgency': 'Within 1 week'
    },
    'Low Risk (17% churn)': {
        'count': len(df[df['risk_segment'] == 'Low Risk']),
        'action': 'Standard engagement — no special spend',
        'trigger': 'Monitor weekly',
        'urgency': 'Routine'
    }
}

for segment, details in strategy.items():
    print(f"\n{segment}")
    print(f"  Captains  : {details['count']:,}")
    print(f"  Action    : {details['action']}")
    print(f"  Trigger   : {details['trigger']}")
    print(f"  Urgency   : {details['urgency']}")

print("\n" + "=" * 60)

   RECOMMENDED INTERVENTION STRATEGY

High Risk (68% churn)
  Captains  : 2,948
  Action    : Immediate Rs.200 bonus + personal outreach call
  Trigger   : Week 4 rides < 8 AND parttime AND bike
  Urgency   : Within 48 hours

Medium Risk (43% churn)
  Captains  : 3,323
  Action    : Push notification with streak bonus offer
  Trigger   : Ride decay > 40% week over week
  Urgency   : Within 1 week

Low Risk (17% churn)
  Captains  : 3,729
  Action    : Standard engagement — no special spend
  Trigger   : Monitor weekly
  Urgency   : Routine



In [12]:
!pip install duckdb -q
import duckdb
con = duckdb.connect()
con.register('captains', df)

city_action = con.execute("""
    SELECT
        city,
        COUNT(*) as total_captains,
        SUM(CASE WHEN risk_segment = 'High Risk' THEN 1 ELSE 0 END) as high_risk,
        ROUND(AVG(churn_probability) * 100, 1) as avg_churn_prob_pct,
        ROUND(AVG(estimated_daily_earnings), 0) as avg_daily_earnings,
        ROUND(SUM(CASE WHEN risk_segment = 'High Risk' THEN 1 ELSE 0 END) * 250.0, 0)
            as retention_budget_needed
    FROM captains
    GROUP BY city
    ORDER BY high_risk DESC
""").df()

print("CITY-LEVEL RETENTION ACTION PLAN")
print("=" * 70)
print(city_action.to_string(index=False))
print("\nNote: retention_budget_needed = high_risk captains x Rs.250 per intervention")

CITY-LEVEL RETENTION ACTION PLAN
     city  total_captains  high_risk  avg_churn_prob_pct  avg_daily_earnings  retention_budget_needed
Bengaluru            3555      873.0                44.7               230.0                 218250.0
Hyderabad            2553      851.0                48.5               234.0                 212750.0
  Chennai            1752      569.0                48.8               230.0                 142250.0
     Pune            1179      376.0                48.8               231.0                  94000.0
    Delhi             961      279.0                47.1               246.0                  69750.0

Note: retention_budget_needed = high_risk captains x Rs.250 per intervention


In [13]:
print(f"""
RAPIDO CAPTAIN CHURN — EXECUTIVE SUMMARY
==========================================

PROBLEM
  40% of new captains go inactive within 4 weeks
  Each lost captain costs Rapido Rs.{revenue_per_saved_captain:,.0f} in lost revenue
  + Rs.500 replacement cost = Rs.{revenue_per_saved_captain + 500:,.0f} total loss per captain

MODEL
  Logistic Regression — AUC 0.739
  Catches 70 out of every 100 churners before they leave
  Segments all captains into Low / Medium / High risk

TOP CHURN SIGNALS (from SHAP analysis)
  1. Rides in Week 4 below 8       (strongest signal)
  2. Estimated daily earnings low
  3. High cancellation rate
  4. No incentive claimed
  5. High zone switching

RECOMMENDED ACTION
  Intervene on {high_risk_captains:,} High Risk captains immediately
  Spend Rs.{total_intervention_cost} per captain = Rs.{total_spend:,.0f} total
  Expected to save {captains_saved:,} captains
  Revenue recovered = Rs.{revenue_recovered:,.0f}
  Net ROI = {roi_pct:.0f}%

CITY PRIORITY
  Hyderabad and Chennai — highest churn probability (48.5-48.8%)
  Bengaluru — highest absolute number of at-risk captains (873)
  Prioritize Bengaluru for volume, Hyderabad/Chennai for rate
""")


RAPIDO CAPTAIN CHURN — EXECUTIVE SUMMARY

PROBLEM
  40% of new captains go inactive within 4 weeks
  Each lost captain costs Rapido Rs.4,860 in lost revenue
  + Rs.500 replacement cost = Rs.5,360 total loss per captain

MODEL
  Logistic Regression — AUC 0.739
  Catches 70 out of every 100 churners before they leave
  Segments all captains into Low / Medium / High risk

TOP CHURN SIGNALS (from SHAP analysis)
  1. Rides in Week 4 below 8       (strongest signal)
  2. Estimated daily earnings low
  3. High cancellation rate
  4. No incentive claimed
  5. High zone switching

RECOMMENDED ACTION
  Intervene on 2,948 High Risk captains immediately
  Spend Rs.250 per captain = Rs.737,000 total
  Expected to save 601 captains
  Revenue recovered = Rs.2,920,860
  Net ROI = 296%

CITY PRIORITY
  Focus on Bengaluru and Chennai first
  Highest concentration of high risk bike captains

